In [133]:
#Path to top-level do-dem directory - edit for your system.
path_to_dodem = '/Users/jmdunca2/do-dem/'
from sys import path as sys_path
sys_path.append(path_to_dodem+'/dodem/')


import nustar_dem_prep as nu
import nustar_utilities as nuutil
import all_nu_analysis as ana

import numpy as np
import pickle
from astropy import units as u
import glob
from astropy.io import fits

import astropy.time

with open('/Users/jmdunca2/do-dem/reference_files/all_targets_postghost_postshut.pickle', 'rb') as f:
    all_targets = pickle.load(f)

with open('/Users/jmdunca2/do-dem/reference_files/all_targets.pickle', 'rb') as f:
    all_targets_old = pickle.load(f)

In [150]:
print(all_targets.keys())
atks = list(all_targets.keys())
print('')
print(all_targets_old.keys())
atkos = list(all_targets_old.keys())


print('')
oldies = [aa for aa in atkos if aa not in atks]
print(oldies)

dict_keys(['01-nov-14_1', '01-nov-14_2', '11-dec-14', '19-feb-16', '22-apr-16_1', '22-apr-16_2', '26-jul-16_1', '27-jul-16_1', '26-jul-16_2', '10-oct-17', '29-may-18_1', '29-may-18_2', '07-sep-18', '09-sep-18', '10-sep-18', '12-apr-19', '13-apr-19', '29-jan-20', '06-jun-20', '07-jun-20', '08-jun-20', '09-jun-20', '08-jan-21', '14-jan-21', '20-jan-21', '29-apr-21', '03-may-21_1', '03-may-21_2', '30-jul-21_1', '30-jul-21_2', '17-nov-21_1', '19-nov-21', '20-nov-21', '22-nov-21_1', '22-nov-21_2', '03-jun-22_1', '03-jun-22_2', '06-sep-22', '09-dec-22', '17-nov-21_2', '21-nov-21', '30-jan-20', '20-jul-21', '24-feb-22', '25-feb-22', '27-feb-22', '18-mar-23_2', '11-dec-22_2'])

dict_keys(['01-sep-15', '02-sep-15', '19-feb-16', '22-apr-16_1', '22-apr-16_2', '26-jul-16_1', '27-jul-16_1', '26-jul-16_2', '11-sep-17', '12-sep-17', '13-sep-17', '10-oct-17', '29-may-18_1', '29-may-18_2', '09-sep-18', '10-sep-18', '12-apr-19', '13-apr-19', '29-jan-20', '06-jun-20', '07-jun-20', '08-jun-20', '09-jun-20

In [146]:

def make_table_inputs(key, all_targets, unused=False):

    import calendar

    # ['January', 'February', 'March', ..., 'December']
    full_months = list(calendar.month_name)[1:]

    ardict = all_targets[key]
    id_dirs = ardict['datapaths']
    orbitlines=[]
    for id in id_dirs:
        evt_data, hdr, obsid = nu.return_submap(datapath=id, fpm='A', return_evt_hdr=True, return_obsid=True)
        time0, time1 = [nuutil.convert_nustar_time(hdr['TSTART']), nuutil.convert_nustar_time(hdr['TSTOP'])]
        #print(type(time0))
        datestring = str(time0.ymdhms.year)+' '+full_months[time0.ymdhms.month-1]+' '+str(time0.ymdhms.day)
        newtopheader = datestring+' &  & & & & & \\\\'
        timerange = [time0, time1]
        hk = glob.glob(id+'/hk/*'+'A_fpm.hk')
            
        #Load in housekeeping data
        hdulist = fits.open(hk[0])
        dat = hdulist[1].data
        hdr = hdulist[1].header
        hdulist.close()

        #Get the livetimes from between the time0,1 found above.
        livetimes = dat['livetime']
        mjd_ref_time=astropy.time.Time(hdr['mjdrefi'],format='mjd')
        livetime_times = astropy.time.Time(mjd_ref_time+dat['time']*u.s,format='mjd')
        durbins = np.where(np.logical_and(livetime_times > time0, livetime_times < time1))[0]
        livetimes_ = np.array(livetimes)[durbins]
        
        total_lvt = str(round(np.sum(livetimes_), 1))
        dur = str(round((time1-time0).to(u.s).value, 1))


        
        neworbitline = '                         & '+timerange[0].strftime('%H:%M')+'-'+timerange[1].strftime('%H:%M')+\
                                ' & '+obsid+' & '+dur+' & '+total_lvt+' & 0 & \\\\'
        orbitlines.append(neworbitline)

    arlines=[]
    for i in range(0, len(ardict['NOAA_ARID'])):
        AR=ardict['NOAA_ARID'][i]
        if not unused:
            qfiles = ardict['res_file_dict(s)'][i]['quiet files all-inst']
            xrtfiles = ardict['res_file_dict(s)'][i]['quiet files aiaxrt']
            #print(len(qfiles))
            if len(qfiles) > 0:
                included='Yes'
            else:
                included='No'
            if not 'Ghost_Ray_Rate_A' in list(ardict.keys()):
                ghoststring='N/A'
            else:
                ghoststring='Yes, corrected.'
            qstring=str(len(qfiles))
            xrtstring=str(len(xrtfiles))
            
        if unused:
            ghoststring='Yes, excluded.'
            qstring='N/A'
            xrtstring='N/A'
            included='No'
            
        #print(ghoststring)
        newarline = '                         & '+AR+' & '+qstring+' & '+ghoststring+' & '+xrtstring+' & '+included+' & \\\\'
        arlines.append(newarline)

    
    
    

    return orbitlines, arlines, newtopheader


    
orbitlines, arlines, newtopheader = make_table_inputs(atks[4], all_targets)

In [144]:
file = 'table_ex3.txt'

with open(file, "r+") as f:
    contents = f.read()
    f.close()
contentslist = contents.split('\n')
count=0
for cc in contentslist:
    print(count, cc)
    count+=1
topheader, midrule, pointingheader, cmidrule, orbitline = contentslist[2:7]
bufferline = contentslist[10]
arheader = contentslist[12]
arline = contentslist[14]
toprule = contentslist[28]


print('')
print(topheader)
print(midrule)

print(pointingheader)
print(cmidrule)
print(orbitline)
print(bufferline)

print(cmidrule)
print(arheader)
print(cmidrule)
print(arline)
print(bufferline)

print(toprule)
print('')

compiled_list=[]
for i in range(0, len(atks)):
    orbitlines, arlines, newtopheader = make_table_inputs(atks[i], all_targets)
    compiled_list.extend([newtopheader, midrule, pointingheader, cmidrule])
    compiled_list.extend(orbitlines)
    compiled_list.extend([bufferline, cmidrule, arheader, cmidrule])
    compiled_list.extend(arlines)
    compiled_list.extend([bufferline, toprule])
    # print(newtopheader)
    # print(midrule)
    # print(pointingheader)
    # print(cmidrule)
    # for oo in orbitlines:
    #     print(oo)
    # print(bufferline)
    # print(cmidrule)
    # print(arheader)
    # for ar in arlines:
    #     print(ar)
    # print(bufferline)
    # print(toprule)
    # print('')
    

compiled_content = ('\n').join(compiled_list)
print(compiled_content)

with open("all_pointings_so_far.txt", "w") as file:
    file.write(compiled_content)
    file.close()

0 
1 
2 2016 July 26/27 &  & & & & & \\
3  \midrule{}
4 Pointing 1:  & Times       & OBSIDs &  Duration (s)  &  Livetime (s) &  XRT Files &  \\* 
5  \cmidrule{2-7}
6 			 & 19:22-20:23 &  20201001001 & 0.0 & 0 &             &        \\
7                 	& 20:59-21:55 & 20201002001  &      0.0      &  0          &                       &            \\ 
8 			& 22:52–23:16 & 20201003001  & 	0.0	&	0	&		& \\
9 			& 00:13–01:13$^{+1}$ & 20201004001  & 0.0	&	0	&		& \\
10 			& & & & & &\\	
11 \cmidrule{2-7}	
12 			&  ARs & Quiescent Times & Ghost Rays & XRT Quiet Times & Included? &  \\	
13 \cmidrule{2-7}
14 			 &AR12568 & 5 & N/A & No& Yes & \\	
15 			 &AR12569 & 1 & N/A & No & Yes & \\		
16 			 & & & & & &\\		
17 \midrule		
18 
19 Pointing 3: & Times       & OBSIDs &  Duration (s)  &  Livetime (s) &  XRT Files &  \\* 
20  \cmidrule{2-7}
21 		&  23:27–23:36&  20201006001 & 0.0 & 0 &            &         \\
22 		& & & & & &\\	
23 \cmidrule{2-7}	
24 			&  ARs & Quiescent Times & Ghost Rays & XR

In [151]:
file = 'table_ex3.txt'

with open(file, "r+") as f:
    contents = f.read()
    f.close()
    
contentslist = contents.split('\n')
count=0
for cc in contentslist:
    print(count, cc)
    count+=1
topheader, midrule, pointingheader, cmidrule, orbitline = contentslist[2:7]
bufferline = contentslist[10]
arheader = contentslist[12]
arline = contentslist[14]
toprule = contentslist[28]


print('')
print(topheader)
print(midrule)

print(pointingheader)
print(cmidrule)
print(orbitline)
print(bufferline)

print(cmidrule)
print(arheader)
print(cmidrule)
print(arline)
print(bufferline)

print(toprule)
print('')

compiled_list=[]
for i in range(0, len(oldies)):
    orbitlines, arlines, newtopheader = make_table_inputs(oldies[i], all_targets_old, unused=True)
    compiled_list.extend([newtopheader, midrule, pointingheader, cmidrule])
    compiled_list.extend(orbitlines)
    compiled_list.extend([bufferline, cmidrule, arheader, cmidrule])
    compiled_list.extend(arlines)
    compiled_list.extend([bufferline, toprule])
    # print(newtopheader)
    # print(midrule)
    # print(pointingheader)
    # print(cmidrule)
    # for oo in orbitlines:
    #     print(oo)
    # print(bufferline)
    # print(cmidrule)
    # print(arheader)
    # for ar in arlines:
    #     print(ar)
    # print(bufferline)
    # print(toprule)
    # print('')
    

compiled_content = ('\n').join(compiled_list)
print(compiled_content)

with open("all_pointings_so_far_old.txt", "w") as file:
    file.write(compiled_content)
    file.close()

0 
1 
2 2016 July 26/27 &  & & & & & \\
3  \midrule{}
4 Pointing 1:  & Times       & OBSIDs &  Duration (s)  &  Livetime (s) &  XRT Files &  \\* 
5  \cmidrule{2-7}
6 			 & 19:22-20:23 &  20201001001 & 0.0 & 0 &             &        \\
7                 	& 20:59-21:55 & 20201002001  &      0.0      &  0          &                       &            \\ 
8 			& 22:52–23:16 & 20201003001  & 	0.0	&	0	&		& \\
9 			& 00:13–01:13$^{+1}$ & 20201004001  & 0.0	&	0	&		& \\
10 			& & & & & &\\	
11 \cmidrule{2-7}	
12 			&  ARs & Quiescent Times & Ghost Rays & XRT Quiet Times & Included? &  \\	
13 \cmidrule{2-7}
14 			 &AR12568 & 5 & N/A & No& Yes & \\	
15 			 &AR12569 & 1 & N/A & No & Yes & \\		
16 			 & & & & & &\\		
17 \midrule		
18 
19 Pointing 3: & Times       & OBSIDs &  Duration (s)  &  Livetime (s) &  XRT Files &  \\* 
20  \cmidrule{2-7}
21 		&  23:27–23:36&  20201006001 & 0.0 & 0 &            &         \\
22 		& & & & & &\\	
23 \cmidrule{2-7}	
24 			&  ARs & Quiescent Times & Ghost Rays & XR

In [156]:
demfolder = '/Users/jmdunca2/do-dem/DEM_folders/'

nustar_folder = '/Users/jmdunca2/nustar/'

#FLARING THE WHOLE TIME
#August 2017

dictt={}


ARdict = {'NOAA_ARID': ['AR12671'],
          'datapaths': ['/Users/jmdunca2/nustar/aug-2017/20312001001/',
                        '/Users/jmdunca2/nustar/aug-2017/20312002001/'],
          'obsids': ['20312001001', '20312002001'],
          'working_dir': '/Users/jmdunca2/do-dem/DEM_folders/initial_dem_aug17/',
          'notes': 'flaring disk AR (Glesener et al. 2020, Duncan et al. 2021, Great American Eclipse',
          'goes_satellite': 16}

dictt['21-aug-17']=ARdict




#March 2023

ARdict = {'NOAA_ARID': ['AR13247,AR13250,AR13254'],
          'HARP': [],
            'datapaths': ['/Users/jmdunca2/nustar/mar-2023/20801021001/',
                         '/Users/jmdunca2/nustar/mar-2023/20801022001/',
                         '/Users/jmdunca2/nustar/mar-2023/20801023001/',
                         '/Users/jmdunca2/nustar/mar-2023/20801024001/',
                         '/Users/jmdunca2/nustar/mar-2023/20801026001/',
                         '/Users/jmdunca2/nustar/mar-2023/20801027001/'],
          'obsids': ['20801021001','20801022001','20801023001',
                     '20801024001','20801026001', '20801027001'],
          'working_dir': demfolder+'/initial_dem_18mar23_1/',
          'ghost_ray_flag': True,
          'loc': ['limb'],
          'hale_class': [''],
          'method': 'fit',
         'notes': '',
          'goes_satellite': 16}

dictt['18-mar-23_1'] = ARdict






keylist = ['21-aug-17', '18-mar-23_1']


file = 'table_ex3.txt'

with open(file, "r+") as f:
    contents = f.read()
    f.close()
    
contentslist = contents.split('\n')
count=0
for cc in contentslist:
    print(count, cc)
    count+=1
topheader, midrule, pointingheader, cmidrule, orbitline = contentslist[2:7]
bufferline = contentslist[10]
arheader = contentslist[12]
arline = contentslist[14]
toprule = contentslist[28]


print('')
print(topheader)
print(midrule)

print(pointingheader)
print(cmidrule)
print(orbitline)
print(bufferline)

print(cmidrule)
print(arheader)
print(cmidrule)
print(arline)
print(bufferline)

print(toprule)
print('')

compiled_list=[]
for i in range(0, len(keylist)):
    orbitlines, arlines, newtopheader = make_table_inputs(keylist[i], dictt, unused=True)
    compiled_list.extend([newtopheader, midrule, pointingheader, cmidrule])
    compiled_list.extend(orbitlines)
    compiled_list.extend([bufferline, cmidrule, arheader, cmidrule])
    compiled_list.extend(arlines)
    compiled_list.extend([bufferline, toprule])
    # print(newtopheader)
    # print(midrule)
    # print(pointingheader)
    # print(cmidrule)
    # for oo in orbitlines:
    #     print(oo)
    # print(bufferline)
    # print(cmidrule)
    # print(arheader)
    # for ar in arlines:
    #     print(ar)
    # print(bufferline)
    # print(toprule)
    # print('')
    

compiled_content = ('\n').join(compiled_list)
print(compiled_content)

with open("all_pointings_so_far_extra.txt", "w") as file:
    file.write(compiled_content)
    file.close()




0 
1 
2 2016 July 26/27 &  & & & & & \\
3  \midrule{}
4 Pointing 1:  & Times       & OBSIDs &  Duration (s)  &  Livetime (s) &  XRT Files &  \\* 
5  \cmidrule{2-7}
6 			 & 19:22-20:23 &  20201001001 & 0.0 & 0 &             &        \\
7                 	& 20:59-21:55 & 20201002001  &      0.0      &  0          &                       &            \\ 
8 			& 22:52–23:16 & 20201003001  & 	0.0	&	0	&		& \\
9 			& 00:13–01:13$^{+1}$ & 20201004001  & 0.0	&	0	&		& \\
10 			& & & & & &\\	
11 \cmidrule{2-7}	
12 			&  ARs & Quiescent Times & Ghost Rays & XRT Quiet Times & Included? &  \\	
13 \cmidrule{2-7}
14 			 &AR12568 & 5 & N/A & No& Yes & \\	
15 			 &AR12569 & 1 & N/A & No & Yes & \\		
16 			 & & & & & &\\		
17 \midrule		
18 
19 Pointing 3: & Times       & OBSIDs &  Duration (s)  &  Livetime (s) &  XRT Files &  \\* 
20  \cmidrule{2-7}
21 		&  23:27–23:36&  20201006001 & 0.0 & 0 &            &         \\
22 		& & & & & &\\	
23 \cmidrule{2-7}	
24 			&  ARs & Quiescent Times & Ghost Rays & XR

In [118]:
atks[25]

'29-apr-21'

In [131]:
f.truncate()

ValueError: I/O operation on closed file.